# CodeBERT training and energy measurements

Run Notebook 03 first. This notebook trains the frozen `microsoft/codebert-base` embedding model separately on the validated and significant prepared datasets. Inputs are capped at 512 tokens, including tokenizer-added special tokens; pretrained positional embeddings are not extended. Attention-mask mean pooling feeds a class-balanced Logistic Regression head evaluated with five problem-grouped folds.

Training cells measure the complete grouped-training workload, including tokenizer/encoder loading, embedding extraction, grouped model fitting, artifact writes, runtime, process CPU use, peak memory, any supported hardware energy counter, and CodeCarbon estimates when installed. Inference excludes model loading and includes only the sampled embedding and prediction workload. Inference is measured separately on up to 100 held-out pairs for 10 repetitions, then reported per pair and linearly extrapolated to that dataset's pair count. This is an estimate, not a full-dataset inference measurement. If no hardware energy counter is available, direct energy remains unavailable; CodeCarbon values are labeled as estimates rather than measurements. The setup cell creates an external power-meter template for later manual readings.

Run the setup cell once, then both training cells, then the two inference cells. This notebook has not been executed as part of these edits.

In [ ]:
import sys
from pathlib import Path

from IPython.display import display

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "Data" / "Processed").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run Notebook 03 first to create Data/Processed.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from measurement_utils import (
    initialize_codebert_resource_experiment,
    measure_codebert_inference,
    train_and_measure_codebert_dataset,
)

MAX_INPUT_LENGTH = 512
BATCH_SIZE = 8
INFERENCE_SAMPLE_PAIRS = 100
INFERENCE_REPETITIONS = 10
PROCESS_SAMPLING_INTERVAL_SECONDS = 0.1

RESOURCE_STATE = initialize_codebert_resource_experiment(
    ROOT,
    sampling_interval_seconds=PROCESS_SAMPLING_INTERVAL_SECONDS,
)
print("Resource run:", RESOURCE_STATE["run_root"])
print("Hardware energy source:", RESOURCE_STATE["energy_counter"].backend or "unavailable")
print("External power-meter template:", RESOURCE_STATE["manual_meter_template"])

In [ ]:
validated_performance, validated_training_measurements = train_and_measure_codebert_dataset(
    RESOURCE_STATE,
    dataset_name="validated",
    max_length=MAX_INPUT_LENGTH,
    batch_size=BATCH_SIZE,
)
display(validated_performance)
display(validated_training_measurements)

In [ ]:
significant_performance, significant_training_measurements = train_and_measure_codebert_dataset(
    RESOURCE_STATE,
    dataset_name="significant",
    max_length=MAX_INPUT_LENGTH,
    batch_size=BATCH_SIZE,
)
display(significant_performance)
display(significant_training_measurements)

## Sampled inference measurements

Each following cell reuses the saved held-out fold heads, encodes the same deterministic sample of up to 100 pairs, and repeats inference 10 times. The recorded per-pair energy and dataset-scale energy are linear extrapolations from those measured batches. Model/encoder loading is excluded; the encoder embedding and Logistic Regression prediction are included.

In [ ]:
validated_inference_measurements = measure_codebert_inference(
    RESOURCE_STATE,
    dataset_name="validated",
    inference_pairs=INFERENCE_SAMPLE_PAIRS,
    inference_repetitions=INFERENCE_REPETITIONS,
    batch_size=BATCH_SIZE,
)
display(validated_inference_measurements.query("task == 'inference'"))

In [ ]:
significant_inference_measurements = measure_codebert_inference(
    RESOURCE_STATE,
    dataset_name="significant",
    inference_pairs=INFERENCE_SAMPLE_PAIRS,
    inference_repetitions=INFERENCE_REPETITIONS,
    batch_size=BATCH_SIZE,
)
display(significant_inference_measurements.query("task == 'inference'"))